# Whisper Base 模型评估 (未微调)

使用 OpenAI Whisper-base 在验证集上测试，评估未微调模型的效果。

## 1. 环境检查

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, re, json, time, gc
import torch

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

try:
    import librosa
    print(f'librosa: {librosa.__version__}')
except ImportError:
    print('ERROR: librosa 未安装')

try:
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    import transformers
    print(f'transformers: {transformers.__version__}')
except ImportError:
    print('ERROR: transformers 未安装')

import editdistance
print(f'editdistance: OK')

print('\n--- 路径检查 ---')
val_path = os.path.expanduser('~/Desktop/hengdong_asr_trainset/manifests/val.jsonl')
print(f'  验证集: {"OK" if os.path.exists(val_path) else "NOT FOUND"} - {val_path}')

PyTorch: 2.11.0
Device: mps
librosa: 0.11.0
transformers: 5.6.2
editdistance: OK

--- 路径检查 ---
  验证集: OK - /Users/fanhua/Desktop/hengdong_asr_trainset/manifests/val.jsonl


## 2. 工具函数与数据加载

In [2]:
_PUNCT = re.compile(
    r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a'
    r'\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]'
)

def norm_punct(text):
    """去除标点符号（用于 CER 计算）"""
    return _PUNCT.sub('', text)

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print('工具函数就绪')

工具函数就绪


In [3]:
# 加载验证集
DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
VAL_JSONL = os.path.join(DATA_ROOT, 'manifests/val.jsonl')

samples = []
with open(VAL_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        audio = s.get('audio_filepath') or s.get('source')
        if audio and os.path.exists(audio):
            samples.append(s)

print(f'验证集: {len(samples)} 条有效样本')
print(f'\n前3条:')
for s in samples[:3]:
    audio = s.get('audio_filepath') or s.get('source')
    text = s.get('text') or s.get('target')
    print(f'  {os.path.basename(audio)} -> {text}')

验证集: 2261 条有效样本

前3条:
  backend_user_1769774772513_1_1.wav -> 爱
  backend_user_1769774772513_20_20.wav -> 东西
  backend_user_1769774772513_24_24.wav -> 多


## 3. 加载 Whisper Base 模型

In [4]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa

print('加载 Whisper-base (未微调)...')
start = time.time()

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-base",
)

# 使用 forced_decoder_ids 确保中文强制解码（避免 MPS language detection 问题）
forced_decoder_ids = processor.get_decoder_prompt_ids(language="chinese", task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base",
)
model = model.to(device)
model.eval()

print(f'模型加载完成, 耗时: {time.time() - start:.1f}s')

加载 Whisper-base (未微调)...


Loading weights: 100%|██████████| 245/245 [00:00<00:00, 8553.89it/s]


模型加载完成, 耗时: 6.1s


## 4. 评估函数

In [5]:
import librosa

def eval_whisper(model, processor, samples, label, forced_decoder_ids):
    """评估 Whisper 模型"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio = s.get('audio_filepath') or s.get('source')
            expected_raw = s.get('text') or s.get('target')
            if not audio or not os.path.exists(audio):
                continue
            
            try:
                # 加载音频
                array, sr = librosa.load(audio, sr=16000)
                
                # 处理输入（transformers 5.x 使用 raw_speech=）
                input_features = processor.feature_extractor(
                    raw_speech=array,
                    sampling_rate=sr,
                    return_tensors="pt",
                ).input_features.to(device)
                
                # 生成（使用 forced_decoder_ids 强制中文）
                generated_ids = model.generate(
                    input_features=input_features,
                    forced_decoder_ids=forced_decoder_ids,
                )
                
                # 解码
                pred_raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            except Exception as e:
                pred_raw = ''
                print(f'Error at {i}: {e}')
            
            # 标点归一化后计算 CER
            expected = norm_punct(expected_raw)
            pred = norm_punct(pred_raw)
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            
            results.append({
                'id': i,
                'expected_raw': expected_raw,
                'predicted_raw': pred_raw,
                'expected': expected,
                'predicted': pred,
                'cer': cer
            })
            
            if (i + 1) % 50 == 0:
                print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {
        'name': label,
        'results': results,
        'cer': cer,
        'exact': exact,
        'total': len(results),
        'time': elapsed
    }

print('评估函数就绪')

评估函数就绪


## 5. 评估 Whisper-base

In [6]:
print('评估 Whisper-base (未微调)...')
whisper_base_res = eval_whisper(model, processor, samples, 'Whisper-base', forced_decoder_ids)

print(f'\n完成! CER={whisper_base_res["cer"]:.2%} | exact={whisper_base_res["exact"]}/{whisper_base_res["total"]} | 耗时={whisper_base_res["time"]:.1f}s')

评估 Whisper-base (未微调)...


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressToken

    Whisper-base: 50/2261 | CER=113.41% | exact=2
    Whisper-base: 100/2261 | CER=109.76% | exact=4
    Whisper-base: 150/2261 | CER=110.00% | exact=5
    Whisper-base: 200/2261 | CER=109.30% | exact=5
    Whisper-base: 250/2261 | CER=109.34% | exact=7
    Whisper-base: 300/2261 | CER=113.67% | exact=8
    Whisper-base: 350/2261 | CER=194.28% | exact=9
    Whisper-base: 400/2261 | CER=183.77% | exact=9
    Whisper-base: 450/2261 | CER=175.54% | exact=9
    Whisper-base: 500/2261 | CER=169.11% | exact=9
    Whisper-base: 550/2261 | CER=183.62% | exact=9
    Whisper-base: 600/2261 | CER=150.84% | exact=9
    Whisper-base: 650/2261 | CER=168.88% | exact=9
    Whisper-base: 700/2261 | CER=159.07% | exact=10
    Whisper-base: 750/2261 | CER=157.31% | exact=10
    Whisper-base: 800/2261 | CER=154.70% | exact=11
    Whisper-base: 850/2261 | CER=152.86% | exact=12
    Whisper-base: 900/2261 | CER=151.09% | exact=13
    Whisper-base: 950/2261 | CER=145.69% | exact=14
    Whisper-base: 1000/226

## 6. 结果汇总

In [7]:
print('=' * 60)
print('Whisper Base 模型评估结果 (验证集, 标点归一化)')
print('=' * 60)
print(f'CER:           {whisper_base_res["cer"]:.2%}')
print(f'精确匹配率:     {whisper_base_res["exact"]}/{whisper_base_res["total"]} ({whisper_base_res["exact"]/whisper_base_res["total"]:.1%})')
print(f'推理耗时:       {whisper_base_res["time"]:.1f}s')
print(f'速度:          {whisper_base_res["total"]/whisper_base_res["time"]:.1f} 条/秒')
print('=' * 60)

Whisper Base 模型评估结果 (验证集, 标点归一化)
CER:           114.10%
精确匹配率:     27/2261 (1.2%)
推理耗时:       326.5s
速度:          6.9 条/秒


## 7. 保存结果

In [8]:
REPORT_DIR = os.path.expanduser('~/Projects/Agent/local')

save_path = os.path.join(REPORT_DIR, 'whisper_base_result.json')
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model': 'Whisper-base (未微调)',
        'cer': round(whisper_base_res['cer'], 4),
        'exact_match': whisper_base_res['exact'],
        'total': whisper_base_res['total'],
        'exact_rate': round(whisper_base_res['exact'] / max(whisper_base_res['total'], 1), 4),
        'time': round(whisper_base_res['time'], 1),
        'speed': round(whisper_base_res['total'] / whisper_base_res['time'], 1),
        'samples': [
            {
                'id': r['id'],
                'expected': r['expected'],
                'expected_raw': r['expected_raw'],
                'predicted': r['predicted'],
                'predicted_raw': r['predicted_raw'],
                'cer': round(r['cer'], 4)
            }
            for r in whisper_base_res['results']
        ]
    }, f, ensure_ascii=False, indent=2)

print(f'结果已保存: {save_path}')

结果已保存: /Users/fanhua/Projects/Agent/local/whisper_base_result.json


## 8. 清理内存

In [9]:
del model, processor
free_memory()
print('内存已清理')

内存已清理
